# EDA On `combine_results.csv` Dataset

Data has 1 row per player in rookie cohort.

Each row explains how the rookie did in various Combine exercise (40-yard dash, three cone drill, etc.). It also tells how NGS scored them (athleticism score, college production score, final score).

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load Data
# ==========================================

combine_results_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "combine_results.csv"), 
    )
)



# ==========================================
# Clean Data
# ==========================================

# ------------------------------------------
# Set all values of "NA" to None
# ------------------------------------------

na_columns = [col for col in combine_results_df.columns if combine_results_df[col].cast(pl.String).eq("NA").any()]

combine_results_df = (
    combine_results_df
    .with_columns(
        pl.col(na_columns).replace("NA", None)
    )
)

# ------------------------------------------
# Convert Numeric Columns From String To 
# Int. These Are Likely String Because Of 
# The "NA" Values From Last Step
# ------------------------------------------

str_to_float_columns = [
    "ten_yd_split",
    "forty",
    "vertical",
    "three_cone",
    "short_shuttle",
]

str_to_int_columns = [
    "broad_jump", 
    "bench_reps", 
    "ngs_college_production_score", 
    "ngs_final_score"
]

combine_results_df = (
    combine_results_df
    .with_columns(
        pl.col(str_to_int_columns).cast(pl.Int64), 
        pl.col(str_to_float_columns).cast(pl.Float64)
    )
)

print("Combine Results Dataset:")
print(f"     - Dimensions: {combine_results_df.shape}")
combine_results_df

In [ ]:
# =======================================
# Missing Data
# =======================================

combine_results_df.select(pl.all().is_null().sum())

In [ ]:
# =======================================
# Missing Data
# =======================================

# NOTE: We see that some drills have a lot of missing data.
#       This could be because some positions don't participate
#       in certain drills, not necessarily just drop outs.

missing_summary_df = (
    combine_results_df
    .select(
        pl.all().is_null().sum()
    )
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"],
    )
    .with_columns(
        missing_pct = (pl.col("missing_count") / combine_results_df.height * 100).round(2)
    )
    .sort("missing_pct", descending=True)
)

print("Columns With Missing Data:")
missing_summary_df.filter(pl.col("missing_count") > 0)

In [ ]:
# =======================================
# Missing Data (CONTINUED)
# =======================================

exercises = [
    "three_cone",
    "bench_reps",
    "short_shuttle",
    "broad_jump",
    "forty",
    "ten_yd_split",
    "vertical",
]

# Calculate the percentage of missing values by position and exercise
missing_pct_df = (
    combine_results_df
    .group_by("combine_position")
    .agg(
        [
            (pl.col(col).is_null().mean() * 100).alias(col)
            for col in exercises
        ]
    )
    .unpivot(
        index="combine_position",
        on=exercises,
        variable_name="exercise",
        value_name="missing_pct",
    )
)

# Create the faceted bar plot
fig = px.bar(
    missing_pct_df,
    x="combine_position",
    y="missing_pct",
    facet_col="exercise",
    facet_col_wrap=3,
    category_orders={
        "combine_position": ["OL", "DL", "WR", "DB", "TE"],
        "exercise": exercises,
    },
    title="Percentage of Missing Combine Exercise Data by Position",
    labels={
        "combine_position": "Position",
        "missing_pct": "Missing Data (%)",
        "exercise": "Exercise",
    },
)

fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.update_layout(
    height=800,
    width=1100,
    showlegend=False,
)

fig.show()

In [ ]:
# =========================================
# nfl_id
# =========================================

print(f"  - Number of Unique Player IDs: {combine_results_df.select("nfl_id").n_unique()}")

In [ ]:
# =========================================
# combine_position
# =========================================

print(f"  - Number of Unique Positions in Raw Data: {combine_results_df.select("combine_position").n_unique()}")
position_counts = combine_results_df.group_by("combine_position").agg(count = pl.len()).sort("combine_position")

fig = px.bar(
    position_counts, 
    x = "combine_position", 
    y = "count", 
    title = "Position Counts Among Rookie Cohort"
)

fig.show()

In [ ]:
# ===========================================
# combine_height
# ===========================================

fig = px.violin(
    combine_results_df, 
    y = "combine_height", 
    x = "combine_position", 
    color = "combine_position",
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# combine_weight
# ===========================================

fig = px.violin(
    combine_results_df, 
    y = "combine_weight", 
    x = "combine_position", 
    color = "combine_position",
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# hand_size
# ===========================================

fig = px.violin(
    combine_results_df, 
    y = "hand_size", 
    x = "combine_position", 
    color = "combine_position",
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# arm_length
# ===========================================

fig = px.violin(
    combine_results_df, 
    y = "arm_length", 
    x = "combine_position", 
    color = "combine_position",
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# wing_span
# ===========================================

fig = px.violin(
    combine_results_df, 
    y = "wing_span", 
    x = "combine_position", 
    color = "combine_position",
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# ten_yd_split
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "ten_yd_split", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# forty
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "ten_yd_split", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# broad_jump
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "broad_jump", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# three_cone
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "three_cone", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# short_shuttle
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "short_shuttle", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# bench_reps
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "bench_reps", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# ngs_athleticism_score
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "ngs_athleticism_score", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# ngs_college_production_score
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "ngs_college_production_score", 
    color = "combine_position", 
    points = "all"
)

fig.show()

In [ ]:
# ===========================================
# ngs_final_score
# ===========================================

fig = px.violin(
    combine_results_df, 
    x = "combine_position", 
    y = "ngs_final_score", 
    color = "combine_position", 
    points = "all"
)

fig.show()